# ch16_networks_sampling

*Computers in the Loop: Digital Control from Sampling to Networked Agents*: EECE 5610 lecture notes prepared and revised by Milad Siami.

Chapters 1–11 of the notes are based on the lecture notes of Prof. Mario Sznaier; all credits are in `ATTRIBUTION.md`.



Run the cells from top to bottom (in Colab: *Runtime > Run all*). Figures are also saved to a `figures` folder.


In [ ]:
#!/usr/bin/env python3
"""EECE 5610, Chapter 16 companion: how fast must networked agents sample?

Run it with
    python3 ch16_networks_sampling.py
It prints the main numbers of the chapter and saves six figures (PNG) in a folder
ch16_figures/ next to this file:

  0. ch16_agreement.png     three agents in a line agree by hand (T = 0.5) or not (T = 0.8)
  1. ch16_roots.png         Euler roots of one agent as T grows (alpha = beta = 1 and 5)
  2. ch16_tmax_vs_lmax.png  largest stable T against the largest Laplacian eigenvalue
  3. ch16_fast_slow.png     positions of five agents: fast and slow agents, sparse and dense graphs
  4. ch16_density.png       largest stable T as links are added, path to complete, 8 agents
  5. ch16_crossing.png      steady-state disagreement under noise: ring against complete graph

Model (forward Euler, as in the notes): x(k+1) = x + T v, v(k+1) = v + T m, with
m = -Lx x - Lv v. Requires numpy, scipy and matplotlib. Change the CAPITAL constants to experiment.
"""

In [ ]:
import itertools
import math
import os

import numpy as np
from scipy import linalg, optimize

import matplotlib
import matplotlib.pyplot as plt

HERE = os.getcwd()
OUT = os.path.join(HERE, "ch16_figures")
os.makedirs(OUT, exist_ok=True)
SPARSE, DENSE = "#1D5FA8", "#C8102E"          # the notes' blue and red
AGENTS = ["#1D5FA8", "#C8102E", "#2E7D32", "#E67E22", "#1F2A33"]   # agents 1-5, as in the notes

In [ ]:
def save(fig, name):
    path = os.path.join(OUT, name)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"  saved {path}")

In [ ]:
# ---------------------------------------------------------------------------------------------
# Graphs and Laplacians
# ---------------------------------------------------------------------------------------------
def laplacian(edges, n):
    L = np.zeros((n, n))
    for i, j in edges:
        L[i, i] += 1
        L[j, j] += 1
        L[i, j] -= 1
        L[j, i] -= 1
    return L

In [ ]:
def four_graphs(n):
    return {"path": [(i, i + 1) for i in range(n - 1)],
            "ring": [(i, (i + 1) % n) for i in range(n)],
            "star": [(0, i) for i in range(1, n)],
            "complete": list(itertools.combinations(range(n), 2))}

In [ ]:
# ---------------------------------------------------------------------------------------------
# One agent (one mode): the bounds and the variances
# ---------------------------------------------------------------------------------------------
def tmax_euler(alpha, beta):
    """Largest stable T of x(k+1) = x + T v, v(k+1) = v - T(alpha x + beta v)."""
    if beta ** 2 < 4 * alpha:
        return beta / alpha                      # complex pair leaves through |z| = 1
    return 4 / (beta + math.sqrt(beta ** 2 - 4 * alpha))   # real root leaves through z = -1

In [ ]:
def tmax_exact(alpha, beta):
    """Same feedback, exact zero-order-hold model (Gamma = [T^2/2, T])."""
    return min(2 / beta, 2 * beta / alpha)

In [ ]:
def mode_variances(alpha, beta, T):
    """Steady-state variances (p11, p22) of one mode under unit-intensity noise."""
    den = (beta - T * alpha) * (4 - 2 * T * beta + T * T * alpha)
    return (2 - T * beta + T * T * alpha) / (alpha * den), 2 / den

In [ ]:
def network(Lx, Lv, T):
    n = Lx.shape[0]
    return np.block([[np.eye(n), T * np.eye(n)], [-T * Lx, np.eye(n) - T * Lv]])

In [ ]:
def disagreement_radius(Lx, Lv, T):
    """Largest root size of the network once the common motion (mode 1) is removed."""
    n = Lx.shape[0]
    lam, U = np.linalg.eigh(Lx)
    W = U[:, 1:]                                  # orthonormal basis orthogonal to the ones vector
    P = np.kron(np.eye(2), W)
    return max(abs(np.linalg.eigvals(P.T @ network(Lx, Lv, T) @ P)))

In [ ]:
def largest_stable_T(Lx, Lv, hi=3.0):
    """Bisection on the root size: an independent check of the closed-form bound."""
    Ts = np.linspace(1e-4, hi, 3000)
    bad = next(T for T in Ts if disagreement_radius(Lx, Lv, T) >= 1)
    lo, up = 1e-4, bad
    for _ in range(60):
        mid = (lo + up) / 2
        lo, up = (mid, up) if disagreement_radius(Lx, Lv, mid) < 1 else (lo, mid)
    return lo

In [ ]:
# ---------------------------------------------------------------------------------------------
# 0. Agreement by hand: agents that set their own velocity, x(k+1) = x(k) - T L x(k)
# ---------------------------------------------------------------------------------------------
print("Two agents, one link, from x = (4, 0):")

In [ ]:
for T in [0.25, 1.0, 1.25]:
    x = np.array([4.0, 0.0])
    rows = [x.copy()]
    for _ in range(3):
        x = x - T * laplacian([(0, 1)], 2) @ x
        rows.append(x.copy())
    print(f"  T = {T}: " + "  ".join(f"({a:g}, {b:g})" for a, b in rows) + f"   gap factor 1 - 2T = {1 - 2 * T:g}")

In [ ]:
L3 = laplacian([(0, 1), (1, 2)], 3)
print(f"Three agents in a line: eigenvalues of L = {np.round(np.linalg.eigvalsh(L3), 6)}; agreement for T < 2/3")
fig, axs = plt.subplots(1, 2, figsize=(9, 3.2))

In [ ]:
for ax, T in zip(axs, [0.5, 0.8]):
    x = np.array([3.0, 0.0, 0.0])
    X = [x.copy()]
    for _ in range(10):
        x = x - T * L3 @ x
        X.append(x.copy())
    X = np.array(X)
    for i in range(3):
        ax.plot(X[:, i], "-o", ms=3, color=AGENTS[i], label=f"agent {i + 1}")
    ax.set_title(f"three agents in a line, T = {T}", fontsize=10)
    ax.set_xlabel("k")
    ax.set_ylim((-12, 12) if T > 0.6 else (-1, 3.4))

In [ ]:
axs[0].legend(fontsize=8)
save(fig, "ch16_agreement.png")
print("Two masses with a spring and a damper (alpha = beta = 2): Tmax =", tmax_euler(2, 2))

In [ ]:
# ---------------------------------------------------------------------------------------------
# 1. One agent: the bound and where the roots go
# ---------------------------------------------------------------------------------------------
print("One agent, m = -alpha x - beta v")

In [ ]:
for a, b in [(1, 1), (5, 5), (2, 1), (2, 4)]:
    print(f"  alpha = {a}, beta = {b}:  Tmax Euler = {tmax_euler(a, b):.4f},"
          f"  exact model = {tmax_exact(a, b):.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 4.6))
th = np.linspace(0, 2 * math.pi, 300)
ax.plot(np.cos(th), np.sin(th), color="0.2", lw=1)

In [ ]:
for (a, b), col, Tend in [((1, 1), SPARSE, 1.0), ((5, 5), DENSE, tmax_euler(5, 5))]:
    for s in np.roots([1, b, a]):                  # Euler roots move as z = 1 + T s
        Tline = np.linspace(0, Tend, 50)
        z = 1 + Tline * s
        ax.plot(z.real, z.imag, color=col, lw=2.5)
        ax.plot(z[-1].real, z[-1].imag, "o", color=col)

In [ ]:
ax.set_aspect("equal")
ax.axhline(0, color="0.6", lw=0.6)
ax.axvline(0, color="0.6", lw=0.6)
ax.set_title("Euler roots z = 1 + Ts: blue alpha = beta = 1 (to T = 1),\n"
             "red alpha = beta = 5 (to T = 0.553)", fontsize=9)
save(fig, "ch16_roots.png")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 2. Five agents, four graphs, L_x = L_v = L
# ---------------------------------------------------------------------------------------------
N_AGENTS = 5

In [ ]:
G5 = four_graphs(N_AGENTS)
print(f"\n{N_AGENTS} agents, Lx = Lv = L")
print("  graph      links  dmax  lambda_max  Tmax(formula)  Tmax(bisection)  exact model  platoon")
EIG = {}

In [ ]:
for name, edges in G5.items():
    L = laplacian(edges, N_AGENTS)
    lam = np.sort(np.linalg.eigvalsh(L))
    EIG[name] = lam
    t_form = min(tmax_euler(l, l) for l in lam[1:])
    t_bis = largest_stable_T(L, L)
    t_exact = min(tmax_exact(l, l) for l in lam[1:])
    t_plat = min(tmax_euler(l + 1, l + 1) for l in lam)
    print(f"  {name:9s}  {len(edges):5d}  {int(L.diagonal().max()):4d}  {lam[-1]:10.4f}"
          f"  {t_form:13.4f}  {t_bis:15.4f}  {t_exact:11.4f}  {t_plat:7.4f}")
    print(f"             eigenvalues {np.round(lam, 4)}")

In [ ]:
lm = np.linspace(0.05, 10, 400)
MARK = {"path": "o", "ring": "^", "star": "s", "complete": "D"}
fig = plt.figure(figsize=(7.2, 5.6))
gs = fig.add_gridspec(2, 4, height_ratios=[1, 2.3], hspace=0.45, wspace=0.15)

In [ ]:
# top row: the four graphs, drawn with the same five agents
pos = {i: (np.cos(np.pi / 2 + 2 * np.pi * i / N_AGENTS), np.sin(np.pi / 2 + 2 * np.pi * i / N_AGENTS))
       for i in range(N_AGENTS)}

In [ ]:
for k, (name, edges) in enumerate(G5.items()):
    g = fig.add_subplot(gs[0, k])
    for i, j in edges:
        g.plot([pos[i][0], pos[j][0]], [pos[i][1], pos[j][1]], color="0.35", lw=1.2, zorder=1)
    deg = laplacian(edges, N_AGENTS).diagonal()
    for i in range(N_AGENTS):
        busiest = deg[i] == deg.max()
        g.scatter(*pos[i], s=70, zorder=2, color=DENSE if busiest else "white", edgecolor="k", lw=1)
    l = EIG[name][-1]
    g.set_title(f"{name}  {MARK[name] and ''}\n{len(edges)} links, " + r"$\lambda_{\max}$" + f" = {l:.2f}\n"
                + r"$T_{\max}$" + f" = {tmax_euler(l, l):.3f} s", fontsize=8)
    g.plot(0, -1.55, MARK[name], color="k", ms=6, mfc="none")
    g.set_xlim(-1.35, 1.35); g.set_ylim(-1.75, 1.3); g.set_aspect("equal"); g.axis("off")

In [ ]:
# bottom: the bound against the largest eigenvalue, with each graph marked
ax = fig.add_subplot(gs[1, :])

In [ ]:
ax.plot(lm, [tmax_euler(l, l) for l in lm], color=DENSE, lw=2.5,
        label="Euler model (each agent computes x + Tv)")
ax.plot(lm, [tmax_exact(l, l) for l in lm], "--", color=SPARSE, lw=1.6,
        label="exact model (a mass driven through a hold)")
ax.axvline(4, color="0.6", lw=0.8, ls=":")
ax.text(4.08, 0.05, r"$\lambda_{\max} = 4$", fontsize=8, color="0.4")

In [ ]:
for name, off in [("path", -0.12), ("ring", 0.12), ("star", -0.12), ("complete", 0.12)]:
    l = EIG[name][-1]
    ax.plot(l + off, tmax_euler(l, l), MARK[name], color=DENSE, ms=7, label=None)
    ax.plot(l + off, tmax_exact(l, l), MARK[name], mfc="none", color=SPARSE, ms=7)

In [ ]:
ax.annotate("path and ring\n(same busiest agent: 2 links)", (3.62, 1.0), xytext=(0.6, 0.62),
            fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.6))
ax.annotate("star and complete\n(busiest agent: 4 links)", (5.0, 0.553), xytext=(6.0, 0.75),
            fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.6))
ax.set_ylim(0, 1.25)
ax.set_xlabel(r"largest Laplacian eigenvalue $\lambda_{\max}$")
ax.set_ylabel(r"largest stable period $T_{\max}$ (s)")
ax.legend(fontsize=8, loc="upper right")
ax.set_title(r"$L_x = L_v = L$; filled markers: Euler model, open markers: exact model;"
             "\nred agents are the busiest (largest degree); markers at the same point are nudged apart", fontsize=8.5)
save(fig, "ch16_tmax_vs_lmax.png")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 3. Simulations: fast and slow agents, sparse and dense graphs
# ---------------------------------------------------------------------------------------------
X0 = np.array([2, -1, 1.5, -2.5, 0.0])

In [ ]:
V0 = np.array([0, 0.5, -0.5, 0.3, -0.3])
T_END = 12.0

In [ ]:
def simulate(L, T):
    F = network(L, L, T)
    s = np.r_[X0, V0]
    xs = [s[:N_AGENTS].copy()]
    for _ in range(int(T_END / T + 1e-9)):
        s = F @ s
        xs.append(s[:N_AGENTS].copy())
    return np.arange(len(xs)) * T, np.array(xs)

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(9, 6), sharex=True)
cases = [("ring", 0.25, "ring, T = 0.25 s (fast agents)", 6), ("ring", 1.05, "ring, T = 1.05 s (slow agents)", 6),
         ("path", 0.6, "path (sparse), T = 0.6 s", 8), ("complete", 0.6, "complete (dense), T = 0.6 s", 8)]

In [ ]:
for ax, (name, T, title, ylim) in zip(axs.flat, cases):
    t, X = simulate(laplacian(G5[name], N_AGENTS), T)
    for i in range(N_AGENTS):
        ax.plot(t, X[:, i], "-o", ms=2.5, lw=1, color=AGENTS[i])
    ax.set_ylim(-ylim, ylim)
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.3)

In [ ]:
for ax in axs[1]:
    ax.set_xlabel("t = kT (s)")

In [ ]:
for ax in axs[:, 0]:
    ax.set_ylabel("x_i(kT)")

In [ ]:
save(fig, "ch16_fast_slow.png")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 4. Adding links one at a time, 8 agents
# ---------------------------------------------------------------------------------------------
N8 = 8

In [ ]:
edges = [(i, i + 1) for i in range(N8 - 1)]
order = [(0, N8 - 1)] + [tuple(sorted((i, (i + k) % N8))) for k in (2, 3) for i in range(N8)] \
    + [(i, i + 4) for i in range(N8 // 2)]
seq = [list(edges)]

In [ ]:
for e in order:
    if e not in edges:
        edges.append(e)
        seq.append(list(edges))

In [ ]:
g = lambda l: tmax_euler(l, l)     # L_x = L_v = L
links, tf, tb, lo_b, up_b = [], [], [], [], []

In [ ]:
for E in seq:
    L = laplacian(E, N8)
    lmax, d = np.linalg.eigvalsh(L)[-1], L.diagonal().max()
    links.append(len(E))
    tf.append(g(lmax))
    tb.append(largest_stable_T(L, L))
    lo_b.append(g(2 * d))
    up_b.append(g(d + 1))

In [ ]:
print(f"\n{N8} agents, links added one at a time: largest stable T (formula, bisection)")

In [ ]:
for m_, a_, b_ in zip(links, tf, tb):
    print(f"  {m_:2d} links: {a_:.4f}  {b_:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.fill_between(links, lo_b, up_b, step="mid", color=SPARSE, alpha=0.15,
                label="degree band g(2 dmax) .. g(dmax + 1)")
ax.step(links, tf, where="mid", color=DENSE, lw=1.5)
ax.plot(links, tb, "o", color=DENSE, ms=4, label="largest stable T")
ax.set_xlabel("number of links (path = 7, ring = 8, complete = 28)")
ax.set_ylabel("largest stable T")
ax.set_ylim(0, 1.12)
ax.legend(fontsize=8)
save(fig, "ch16_density.png")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 5. Noise: steady-state disagreement, ring against complete graph
# ---------------------------------------------------------------------------------------------
def V(name, T, which):
    """Sum over modes 2..n of the per-mode variance (which = 0 positions, 1 velocities)."""
    return sum(mode_variances(l, l, T)[which] for l in EIG[name][1:])

In [ ]:
def V_lyapunov(name, T, which):
    """The same number from the discrete Lyapunov equation of the whole disagreement network."""
    L = laplacian(G5[name], N_AGENTS)
    lam, U = np.linalg.eigh(L)
    P2 = np.kron(np.eye(2), U[:, 1:])
    Fd = P2.T @ network(L, L, T) @ P2
    Gd = P2.T @ np.vstack([np.zeros((N_AGENTS, N_AGENTS)), math.sqrt(T) * np.eye(N_AGENTS)])
    P = P2 @ linalg.solve_discrete_lyapunov(Fd, Gd @ Gd.T) @ P2.T
    blk = slice(0, N_AGENTS) if which == 0 else slice(N_AGENTS, 2 * N_AGENTS)
    return np.trace(P[blk, blk])

In [ ]:
print("\nSteady-state disagreement (closed form; Lyapunov equation of the network)")
print("   T     ring Vv   complete Vv   ring Vx   complete Vx")

In [ ]:
for T in [0.1, 0.2, 0.4, 0.5]:
    print(f"  {T:.1f}  {V('ring', T, 1):8.3f}  {V('complete', T, 1):11.3f}  {V('ring', T, 0):8.3f}"
          f"  {V('complete', T, 0):11.3f}   (check: {V_lyapunov('ring', T, 1):.3f},"
          f" {V_lyapunov('complete', T, 1):.3f})")

In [ ]:
cross_v = optimize.brentq(lambda T: V("ring", T, 1) - V("complete", T, 1), 0.2, 0.55)
cross_x = optimize.brentq(lambda T: V("ring", T, 0) - V("complete", T, 0), 0.3, 0.5527)
print(f"  crossing: T = {cross_v:.4f} (velocities), T = {cross_x:.4f} (positions);"
      f" complete graph unstable above {1 - 1 / math.sqrt(5):.4f}")

fig, axs = plt.subplots(1, 2, figsize=(9, 3.8))

In [ ]:
for ax, which, lab in [(axs[0], 1, "velocity disagreement V_v"), (axs[1], 0, "position disagreement V_x")]:
    Tr = np.linspace(0.001, 0.75, 400)
    Tc = np.linspace(0.001, 0.548, 400)
    ax.semilogy(Tr, [V("ring", T, which) for T in Tr], color=SPARSE, lw=2.2, label="ring (sparse)")
    ax.semilogy(Tc, [V("complete", T, which) for T in Tc], color=DENSE, lw=2.2, label="complete (dense)")
    ax.axvline(1 - 1 / math.sqrt(5), color=DENSE, ls="--", lw=1)
    c = cross_v if which == 1 else cross_x
    ax.plot(c, V("ring", c, which), "ko", ms=5)
    ax.set_xlabel("T (s)")
    ax.set_title(lab, fontsize=10)
    ax.set_ylim(0.05, 60)
    ax.grid(alpha=0.3, which="both")

In [ ]:
axs[0].legend(fontsize=8)
save(fig, "ch16_crossing.png")

## Explore




In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
r"""**How fast must the agents sample?** Each agent $i$ updates its position and velocity every
$T$ seconds, $x_i(k+1) = x_i + T v_i$, $v_i(k+1) = v_i + T m_i$, with $m = -L x - L v$ for the graph
Laplacian $L$. A mode with Laplacian eigenvalue $\lambda$ is stable for $T < 1$ when
$\lambda < 4$, and for $T < 4/(\lambda + \sqrt{\lambda^2 - 4\lambda})$ when $\lambda \ge 4$.
Start with the ring of five agents at $T = 0.25$ s, then push $T$ past 1 s. Switch to the complete
graph: more links, a larger $\lambda_{\max}$, and the limit drops to $0.553$ s. On the right, each
mode's roots $z = 1 + Ts$ walk out from $z = 1$ as $T$ grows.
"""
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"
AGENT_COLORS = [ORANGE, BLUE, TEAL, GRAY, "#8E44AD", "#C0392B", "#27AE60", "#2C3E50", "#D4AC0D",
                "#7F8C8D"]
X0 = np.array([2.0, -1.0, 1.5, -2.5, 0.0, 1.0, -1.8, 2.2, -0.5, 0.8])
V0 = np.array([0.0, 0.5, -0.5, 0.3, -0.3, 0.2, -0.4, 0.1, 0.4, -0.2])
T_END = 12.0


def _laplacian(graph, n):
    if graph == "path":
        edges = [(i, i + 1) for i in range(n - 1)]
    elif graph == "ring":
        edges = [(i, (i + 1) % n) for i in range(n)]
    elif graph == "star":
        edges = [(0, i) for i in range(1, n)]
    else:
        edges = [(i, j) for i in range(n) for j in range(i + 1, n)]
    L = np.zeros((n, n))
    for i, j in edges:
        L[i, i] += 1
        L[j, j] += 1
        L[i, j] -= 1
        L[j, i] -= 1
    return L, len(edges)


def _t_limit(lam_x, lam_v):
    """Largest stable T of one mode of the Euler model (second-order test)."""
    if lam_v ** 2 < 4 * lam_x:
        return lam_v / lam_x
    return 4 / (lam_v + np.sqrt(lam_v ** 2 - 4 * lam_x))


def explore(graph="ring", agents=5, T=0.25):
    n = int(agents)
    L, links = _laplacian(graph, n)
    lam = np.sort(np.linalg.eigvalsh(L))
    lam[np.abs(lam) < 1e-9] = 0.0
    limits = [_t_limit(l, l) for l in lam[1:]]         # mode 1 (lambda = 0) is the common motion
    t_max = min(limits)

    F = np.block([[np.eye(n), T * np.eye(n)], [-T * L, np.eye(n) - T * L]])
    s = np.r_[X0[:n], V0[:n] - V0[:n].mean()]
    steps = int(np.floor(T_END / T + 1e-9))
    X = [s[:n].copy()]
    for _ in range(steps):
        s = F @ s
        X.append(s[:n].copy())
    X = np.array(X)
    t = T * np.arange(steps + 1)

    roots = [1 + T * np.roots([1.0, l, l]).astype(complex) for l in lam[1:]]
    r_dis = max(np.max(np.abs(r)) for r in roots)
    stable = r_dis < 1 - 1e-12

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.9, 1]})
    for i in range(n):
        ax1.plot(t, X[:, i], "-o", color=AGENT_COLORS[i], lw=1.2, ms=3 if T >= 0.2 else 1.5)
    ax1.set_xlim(0, T_END)
    lim1 = 4.0 if stable else 8.0
    ax1.set_ylim(-lim1, lim1)
    ax1.set_xlabel("t = kT (s)")
    ax1.set_ylabel("position $x_i(kT)$")
    verdict = "consensus" if stable else "unstable"
    ax1.set_title(f"{graph}, {n} agents, T = {T:g} s: {verdict} (limit {t_max:.3f} s)", fontsize=11)

    th = np.linspace(0, 2 * np.pi, 300)
    ax2.plot(np.cos(th), np.sin(th), color="k", lw=1)
    Tline = np.linspace(0, T, 40)
    for l, r in zip(lam[1:], roots):
        for sp in np.roots([1.0, l, l]).astype(complex):
            z = 1 + Tline * sp
            ax2.plot(z.real, z.imag, color=GRAY, lw=1)
        ax2.plot(r.real, r.imag, "x", color=ORANGE, ms=10, mew=2.2)
    ax2.plot([1], [0], "o", color=BLUE, ms=6)
    ax2.axhline(0, color="k", lw=0.5)
    ax2.axvline(0, color="k", lw=0.5)
    lim = max(1.3, 1.1 * r_dis)
    ax2.set_xlim(-lim, lim)
    ax2.set_ylim(-lim, lim)
    ax2.set_aspect("equal")
    ax2.set_xlabel("Re z")
    ax2.set_ylabel("Im z")
    ax2.set_title("roots of the modes $\\lambda_2, \\dots, \\lambda_n$", fontsize=11)
    fig.tight_layout()

    print(f"{graph}, {n} agents, {links} links:  Laplacian eigenvalues {np.round(lam, 4)}")
    print(f"largest stable T = {t_max:.4f} s (lambda_max = {lam[-1]:.4f})")
    print(f"T = {T:g} s: largest disagreement root |z| = {r_dis:.4f}  ->  "
          + ("stable: the agents reach consensus" if stable else "NOT stable: the agents fly apart"))
    plt.show()


try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    interact(explore,
             graph=Dropdown(options=["path", "ring", "star", "complete"], value="ring",
                            description="graph"),
             agents=IntSlider(value=5, min=3, max=10, step=1, description="agents",
                              continuous_update=False),
             T=FloatSlider(value=0.25, min=0.05, max=1.5, step=0.05, description="T (s)",
                           continuous_update=False))
except ImportError:
    explore()

## Hands-on

**Hands-on, Chapter 16.** These cells go with the Hands-on exercises at the end of
Chapter 16. Do each exercise on paper first and write down your prediction; then use the
matching section below to check it. Each section has sliders: change one thing at a time.


In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"
AGENT_COLORS = [BLUE, ORANGE, TEAL, GRAY, "#B4500A", "#1F2A33", "#7A9CC6", "#C9A227"]

In [ ]:
def laplacian(n, edges):
    L = np.zeros((n, n))
    for i, j in edges:
        L[i, i] += 1
        L[j, j] += 1
        L[i, j] -= 1
        L[j, i] -= 1
    return L

In [ ]:
def graph_edges(name, n):
    if name == "path":
        return [(i, i + 1) for i in range(n - 1)]
    if name == "ring":
        return [(i, (i + 1) % n) for i in range(n)]
    if name == "star":
        return [(0, i) for i in range(1, n)]
    return [(i, j) for i in range(n) for j in range(i + 1, n)]       # complete

In [ ]:
def disagreement_radius(Lx, Lv, T):
    """Spectral radius of the Euler network F restricted to the disagreement modes."""
    n = Lx.shape[0]
    P = np.linalg.qr(np.column_stack([np.ones(n), np.eye(n)[:, : n - 1]]))[0][:, 1:]
    I = np.eye(n - 1)
    F = np.block([[I, T * I], [-T * P.T @ Lx @ P, I - T * P.T @ Lv @ P]])
    return max(abs(np.linalg.eigvals(F)))

In [ ]:
def largest_T(Lx, Lv, Tcap=3.0):
    """First T at which some disagreement mode reaches the unit circle (scan, then bisect)."""
    Ts = np.linspace(1e-3, Tcap, 1500)
    bad = [T for T in Ts if disagreement_radius(Lx, Lv, T) >= 1 - 1e-12]
    if not bad:
        return np.inf
    hi = bad[0]
    lo = hi - (Ts[1] - Ts[0])
    for _ in range(50):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if disagreement_radius(Lx, Lv, mid) < 1 - 1e-12 else (lo, mid)
    return lo

In [ ]:
def simulate(Lx, Lv, T, x0, v0, tend=12.0):
    x, v = np.array(x0, float), np.array(v0, float)
    k = int(round(tend / T))
    X = [x.copy()]
    for _ in range(k):
        x, v = x + T * v, v + T * (-Lx @ x - Lv @ v)
        X.append(x.copy())
    return np.arange(k + 1) * T, np.array(X)

In [ ]:
# ---------------------------------------------------------------- H1
def h1_leader(m=6, T=0.3):
    """H1: six drones in a ring (nodes 1..6) and a leader (node 0) linked to drones 1..m; Lx = Lv = L."""
    edges = [(i, i % 6 + 1) for i in range(1, 7)] + [(0, i) for i in range(1, m + 1)]
    L = laplacian(7, edges)
    lam = np.sort(np.linalg.eigvalsh(L))
    Tm = largest_T(L, L)
    x0 = [0.0, 2, -1, 1.5, -2.5, 0.5, -0.5]
    v0 = [0.0, 0.5, -0.5, 0.3, -0.3, 0.2, -0.2]
    t, X = simulate(L, L, T, x0, v0)
    X = X - X.mean(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(10, 3.6))
    for i in range(7):
        ax.plot(t, X[:, i], "o-", ms=2.5, lw=1, color=AGENT_COLORS[i % 8],
                label="leader" if i == 0 else None)
    ax.set_ylim(-6, 6)
    ax.set_xlabel("t = kT (s)")
    ax.set_ylabel("x_i - average")
    ax.legend(loc="upper right", fontsize=9)
    ax.set_title(f"leader linked to {m}: eigenvalues {np.round(lam, 3)}, Tmax = {Tm:.3f} s; "
                 f"T = {T:g} s", fontsize=10)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H2
def h2_two_rows(T=1.0, zeta=0.5):
    """H2: complete bipartite graph, two rows of three robots; Lv = L, Lx = zeta L."""
    L = laplacian(6, [(i, j) for i in range(3) for j in range(3, 6)])
    lam = np.sort(np.linalg.eigvalsh(L))
    Tm = largest_T(zeta * L, L)
    x0 = [1.0, 0.5, 0.8, -1.0, -0.4, -0.9]
    t, X = simulate(zeta * L, L, T, x0, np.zeros(6), tend=8.0)
    fig, ax = plt.subplots(figsize=(10, 3.6))
    for i in range(6):
        ax.plot(t, X[:, i], "o-", ms=3, lw=1, color=BLUE if i < 3 else ORANGE,
                label=("row 1" if i == 0 else "row 2" if i == 3 else None))
    ax.set_ylim(-5, 5)
    ax.set_xlabel("t = kT (s)")
    ax.legend(loc="upper right", fontsize=9)
    ax.set_title(f"eigenvalues {np.round(lam, 3)}, zeta = {zeta:g}: Tmax = {Tm:.3f} s; "
                 f"T = {T:g} s", fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H3
def h3_two_graphs(position_graph="complete", velocity_graph="path", n=4, T=0.1):
    """H3: springs on one graph, dampers on another; Euler network, largest stable T."""
    Lx = laplacian(n, graph_edges(position_graph, n))
    Lv = laplacian(n, graph_edges(velocity_graph, n))
    shared = np.allclose(Lx @ Lv, Lv @ Lx)
    Tm = largest_T(Lx, Lv)
    Ts = np.linspace(0.01, 1.4, 300)
    rad = [disagreement_radius(Lx, Lv, T_) for T_ in Ts]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8))
    ax1.plot(Ts, rad, color=BLUE, lw=1.6)
    ax1.axhline(1, color="k", lw=0.8, ls="--")
    ax1.plot(T, disagreement_radius(Lx, Lv, T), "o", color=ORANGE, ms=8)
    ax1.set_ylim(0, 2)
    ax1.set_xlabel("T (s)")
    ax1.set_ylabel("largest |z| of the disagreement")
    rng = np.random.default_rng(3)
    t, X = simulate(Lx, Lv, T, rng.uniform(-2, 2, n), rng.uniform(-0.5, 0.5, n))
    X = X - X.mean(axis=1, keepdims=True)
    for i in range(n):
        ax2.plot(t, X[:, i], lw=1, color=AGENT_COLORS[i % 8])
    ax2.set_ylim(-4, 4)
    ax2.set_xlabel("t = kT (s)")
    ax2.set_ylabel("x_i - average")
    fig.suptitle(f"springs: {position_graph}, dampers: {velocity_graph}, n = {n}: Tmax = {Tm:.3f} s"
                 f" ({'shared' if shared else 'no shared'} eigenvectors)", fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H4
def h4_radio_budget(graph="(R) ring", T=0.25):
    """H4: eight robots; (R) ring, (D) ring + diagonals, (S) ring + skip-one links; three tests."""
    n = 8
    edges = [(i, (i + 1) % n) for i in range(n)]
    if graph.startswith("(D)"):
        edges += [(i, i + 4) for i in range(4)]
    if graph.startswith("(S)"):
        edges += [(i, (i + 2) % n) for i in range(n)]
    L = laplacian(n, edges)
    dmax = int(max(np.diag(L)))
    Tm = largest_T(L, L)
    r = disagreement_radius(L, L, T)
    env = r ** (15 / T)
    tests = [T <= 0.8 * Tm + 1e-12, env <= 0.05, dmax / T <= 10 + 1e-12]
    x0 = [2, -1, 1.5, -2.5, 0, 1, -0.5, -0.5]
    v0 = [0, 0.5, -0.5, 0.3, -0.3, 0.2, -0.1, -0.1]
    t, X = simulate(L, L, T, x0, v0, tend=15.0)
    X = X - X.mean(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(10, 3.6))
    for i in range(n):
        ax.plot(t, X[:, i], lw=1, color=AGENT_COLORS[i])
    ax.set_ylim(-4, 4)
    ax.set_xlabel("t = kT (s)")
    ax.set_ylabel("x_i - average")
    yn = ["yes" if ok else "no" for ok in tests]
    ax.set_title(f"{graph}, T = {T:.2f} s: margin {yn[0]} (Tmax {Tm:.3f}), agreement {yn[1]} "
                 f"(|z|^(15/T) = {env:.3g}), radio {yn[2]} ({dmax / T:.1f}/s)", fontsize=10)
    plt.show()

In [ ]:
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    interact(h1_leader, m=IntSlider(value=6, min=1, max=6, description="links m",
                                    continuous_update=False),
             T=FloatSlider(value=0.3, min=0.05, max=1.0, step=0.01, description="T (s)",
                           continuous_update=False))
    interact(h2_two_rows, T=FloatSlider(value=1.0, min=0.05, max=1.2, step=0.01, description="T (s)",
                                        continuous_update=False),
             zeta=FloatSlider(value=0.5, min=0.1, max=2.0, step=0.05, description="zeta",
                              continuous_update=False))
    graphs = ["path", "ring", "star", "complete"]
    interact(h3_two_graphs,
             position_graph=Dropdown(options=graphs, value="complete", description="springs"),
             velocity_graph=Dropdown(options=graphs, value="path", description="dampers"),
             n=IntSlider(value=4, min=3, max=8, description="n", continuous_update=False),
             T=FloatSlider(value=0.1, min=0.02, max=1.2, step=0.01, description="T (s)",
                           continuous_update=False))
    interact(h4_radio_budget,
             graph=Dropdown(options=["(R) ring", "(D) ring + diagonals", "(S) ring + skip-one"],
                            value="(R) ring", description="graph"),
             T=FloatSlider(value=0.25, min=0.05, max=0.8, step=0.05, description="T (s)",
                           continuous_update=False))
except ImportError:
    h1_leader()
    h2_two_rows()
    h3_two_graphs()
    h4_radio_budget()